# MIND-small: canonical data preprocessing

Run from top to bottom to produce the shared CSV handoff. EDA lives in `notebooks/eda/`; modelling starts from the exported CSVs. The helpers in `src/preprocessing.py` contain the common loading, parsing and integrity checks. See README.md for local/Colab setup.

## 1. Configure paths

Raw data is unchanged. Set `MIND_RAW_DIR` for a shared Drive folder and `BT4222_PROJECT_ROOT` if the notebook is outside the clone.

In [1]:
from pathlib import Path
import os
import sys
from IPython.display import display

# In Colab, clone/open the repository first; set BT4222_PROJECT_ROOT if needed.
# For shared Drive data, set MIND_RAW_DIR to the folder containing MINDsmall_train.
try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")

configured_root = os.environ.get("BT4222_PROJECT_ROOT")
if configured_root:
    PROJECT_ROOT = Path(configured_root).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "src/data_loading.py").is_file()), None)
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src/data_loading.py").is_file():
    raise FileNotFoundError("Open the cloned repo, or set BT4222_PROJECT_ROOT to its path.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_ROOT = Path(os.environ.get("MIND_RAW_DIR", str(PROJECT_ROOT / "data/raw"))).expanduser()


## 2. Choose a new output version

The default below explicitly permits a train/validation-only build because dev is not currently included. Set `REQUIRE_TEST=True` for the complete handoff after adding `MINDsmall_dev`. Existing output directories cause an error: use a new version for each rebuild. A present but incomplete dev folder is an error.

In [2]:
OUTPUT_DIR = PROJECT_ROOT / "data/processed/v1"
REQUIRE_TEST = False
print("Raw data:", DATA_ROOT)
print("Output:", OUTPUT_DIR)


Raw data: /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/data/raw
Output: /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/data/processed/v1


## 3. Prepare, validate and export

- Read literal quotes correctly; reject invalid IDs, timestamps, candidate tokens and malformed entity JSON.
- Keep the supplied prior history and every candidate in original order. No negative sampling.
- Train: November 9–12, 2019; validation: November 13–14; test: all dev events, verified strictly later. Split whole events before expanding candidates.
- Retain metadata for candidates and history articles from the appropriate source only. Preserve original text, flag missingness, parse entities and calculate whitespace word lengths. Missing metadata gets explicit placeholders.
- Verify chronology, event uniqueness, candidate reconstruction and metadata joins. Export separate event/candidate/news CSVs with JSON list columns and portable string event keys.

The manifest records source checksums, code checksum, versions and completeness. Files are published only after all partitions pass. Timestamps have no assumed timezone.

In [3]:
from src.preprocessing import build_dataset

split_summary = build_dataset(DATA_ROOT, OUTPUT_DIR, require_test=REQUIRE_TEST)
display(split_summary)


FileExistsError: /Users/roniheikkonen/Desktop/NUS/Y3S1/BT4222/bt4222/data/processed/v1 already exists. Choose a new output version.

## 4. Load the same data teammates will use

`load_partition` restores string IDs, timestamps, booleans and JSON lists. Read the exported `data_dictionary.md` before engineering features. Never use labels or current-event click totals as features. Fit learned transforms on training only; compute interaction statistics from permitted past events.

In [4]:
from src.data_loading import load_partition

events, candidates, news = load_partition("train", OUTPUT_DIR)
display(events.head())
display(candidates.head())
display(news[["news_id", "category", "title", "abstract_missing"]].head())


,impression_id,user_id,time,history,timestamp,source_file,event_key,history_ids,history_length,history_missing
0,20112,U65916,11/9/2019 12:00:19 AM,N51706 N40767 N12096 N9798 N38802 N54827 N5780...,2019-11-09 00:00:19,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:20112,"[N51706, N40767, N12096, N9798, N38802, N54827...",15,False
1,13807,U49985,11/9/2019 12:01:13 AM,N5056 N29975 N53234 N39603 N50032 N8422 N53580...,2019-11-09 00:01:13,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:13807,"[N5056, N29975, N53234, N39603, N50032, N8422,...",93,False
2,27660,U25550,11/9/2019 12:02:44 AM,N17260 N38298 N33976 N47719 N14888 N18870 N4607,2019-11-09 00:02:44,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:27660,"[N17260, N38298, N33976, N47719, N14888, N1887...",7,False
3,152217,U19710,11/9/2019 12:02:50 AM,N3530 N48284 N43019 N62546 N138 N13138 N10676 ...,2019-11-09 00:02:50,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:152217,"[N3530, N48284, N43019, N62546, N138, N13138, ...",8,False
4,42166,U38106,11/9/2019 12:03:09 AM,N16874 N264 N48697 N51366,2019-11-09 00:03:09,MINDsmall_train/behaviors.tsv,MINDsmall_train/behaviors.tsv:42166,"[N16874, N264, N48697, N51366]",4,False


,event_key,candidate_position,news_id,label
0,MINDsmall_train/behaviors.tsv:20112,0,N54300,0
1,MINDsmall_train/behaviors.tsv:20112,1,N46057,1
2,MINDsmall_train/behaviors.tsv:20112,2,N57005,0
3,MINDsmall_train/behaviors.tsv:20112,3,N52154,0
4,MINDsmall_train/behaviors.tsv:20112,4,N57099,0


,news_id,category,title,abstract_missing
0,N10,health,7 Running Workouts for Weight Loss,False
1,N100,sports,Astros finally get timely hitting to get back ...,False
2,N1000,sports,"Sounders vs. Real Salt Lake: Highlights, stats...",False
3,N10000,sports,"The Day in Sports: Wednesday, October 30, 2019",False
4,N10001,news,Vero Beach man charged with battery in connect...,False
